# Parliamentary Gender Gap — Spanish Congress 2015-2023

Real-data analysis of **gendered rhetorical differences** in the Spanish Parliament using
**ParlaMint-ES** (CC-BY 4.0, 32K speeches, 841 MPs) and **ChronosVector (CVX)**.

### Research questions

| RQ | Question | CVX approach |
|----|----------|-------------|
| RQ1 | Do female and male MPs use **different rhetorical registers**? | Anchor projection to rhetorical dimensions |
| RQ2 | Has the **gender gap in rhetoric** changed from 2015 to 2023? | Small multiples over time |
| RQ3 | Do **political events** (COVID, motions of censure) shift the gap? | Velocity, changepoints, counterfactual |
| RQ4 | Is rhetoric more **polarized by party or by gender**? | Cohort divergence, signature distance |
| RQ5 | Can CVX temporal features **predict speaker gender** from rhetoric? | Classification with temporal split |

### Data source

**ParlaMint-ES v5.0** — Spanish Parliament (Congreso de los Diputados), 2015-2023.
32,541 speeches from 841 MPs (355 women, 486 men). CC-BY 4.0.

> Erjavec et al. (2023). "The ParlaMint corpora of parliamentary proceedings." *Language Resources and Evaluation*.

In [1]:
import chronos_vector as cvx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.special import softmax as sp_softmax
from tqdm.auto import tqdm
import glob, os, time, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
CACHE_DIR = f'{DATA_DIR}/cache'
PARLAMINT_DIR = f'{DATA_DIR}/parlamint/ParlaMint-ES.txt'

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

C_FEMALE = '#e74c3c'
C_MALE   = '#3498db'
C_EVENT  = '#f39c12'
C_ACCENT = '#2ecc71'
TEMPLATE = 'plotly_dark'

# Key political events in Spain 2015-2023
KEY_EVENTS = {
    '2016-06-26': 'Elections 2016',
    '2017-10-01': 'Catalan referendum',
    '2018-06-01': 'Motion of censure',
    '2019-04-28': 'Elections Apr 2019',
    '2019-11-10': 'Elections Nov 2019',
    '2020-03-14': 'COVID state of alarm',
    '2020-10-25': 'State of alarm II',
    '2022-02-24': 'Ukraine invasion',
}

print(f'Events: {len(KEY_EVENTS)}')

Events: 8


---
## 1. Load ParlaMint-ES

Load all sessions (2015-2023), join text with metadata, filter to regular MPs with known gender.

In [2]:
# Load and join all ParlaMint-ES sessions
UNIFIED_CACHE = f'{DATA_DIR}/parlamint/parlamint_es_unified.parquet'

if os.path.exists(UNIFIED_CACHE):
    df_all = pd.read_parquet(UNIFIED_CACHE)
    print(f'Loaded cached: {len(df_all):,} speeches')
else:
    all_rows = []
    for year in range(2015, 2024):
        txt_files = sorted(glob.glob(f'{PARLAMINT_DIR}/{year}/*.txt'))
        for txt_file in txt_files:
            base = txt_file[:-4]
            meta_file = base + '-meta-en.tsv'
            if not os.path.exists(meta_file):
                continue
            
            texts_dict = {}
            with open(txt_file, encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('\t', 1)
                    if len(parts) == 2:
                        texts_dict[parts[0]] = parts[1]
            
            meta = pd.read_csv(meta_file, sep='\t', on_bad_lines='skip')
            
            for _, row in meta.iterrows():
                speech_id = row.get('ID', '')
                text = texts_dict.get(speech_id, '')
                if not text or len(text) < 20:
                    continue
                all_rows.append({
                    'speech_id': speech_id,
                    'text': text,
                    'date': row.get('Date', ''),
                    'speaker_name': row.get('Speaker_name', ''),
                    'speaker_gender': row.get('Speaker_gender', '-'),
                    'speaker_party': row.get('Speaker_party_name', ''),
                    'party_status': row.get('Party_status', ''),
                    'speaker_role': row.get('Speaker_role', ''),
                })
    
    df_all = pd.DataFrame(all_rows)
    df_all.to_parquet(UNIFIED_CACHE)
    print(f'Parsed {len(df_all):,} speeches')

# Filter: regular MPs with known gender, then reset index for embedding alignment
df = df_all[(df_all['speaker_role'] == 'Regular') & (df_all['speaker_gender'].isin(['M', 'F']))].copy()
# Store original indices for embedding lookup, then reset
df['orig_idx'] = df.index
df = df.reset_index(drop=True)
df['date'] = pd.to_datetime(df['date'])
df['unix_ts'] = df['date'].apply(lambda x: int(x.timestamp()))
df['year'] = df['date'].dt.year

print(f'\nFiltered: {len(df):,} speeches')
print(f'  Female: {(df["speaker_gender"]=="F").sum():,} ({df[df["speaker_gender"]=="F"]["speaker_name"].nunique()} MPs)')
print(f'  Male: {(df["speaker_gender"]=="M").sum():,} ({df[df["speaker_gender"]=="M"]["speaker_name"].nunique()} MPs)')
print(f'  Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'  Top parties:')
for party, count in df['speaker_party'].value_counts().head(6).items():
    f_pct = (df[df['speaker_party']==party]['speaker_gender']=='F').mean()
    print(f'    {party}: {count:,} speeches ({f_pct:.0%} female)')

Loaded cached: 73,631 speeches

Filtered: 32,291 speeches
  Female: 12,081 (346 MPs)


  Male: 20,210 (476 MPs)
  Date range: 2015-01-20 to 2023-02-23
  Top parties:
    Partido Popular: 6,324 speeches (38% female)
    Partido Socialista Obrero Español: 6,172 speeches (51% female)
    Ciudadanos: 2,405 speeches (36% female)
    Unidas Podemos: 2,193 speeches (43% female)
    Vox: 1,872 speeches (32% female)
    Euzko Alderdi Jeltzalea-Partido Nacionalista Vasco: 1,844 speeches (25% female)


---
## 2. Embed & Build CVX Index

Encode speeches with multilingual sentence-transformers. Aggregate daily per gender
to create two temporal entities: **Female MPs** (entity 1) and **Male MPs** (entity 2).

In [3]:
# Embed ALL speeches (before filtering) so indices align
EMB_CACHE = f'{CACHE_DIR}/parlamint_embeddings.npz'

if os.path.exists(EMB_CACHE):
    all_embeddings = np.load(EMB_CACHE)['embeddings']
    print(f'Loaded cached embeddings: {all_embeddings.shape}')
else:
    from sentence_transformers import SentenceTransformer
    print(f'Encoding {len(df_all):,} speeches with {MODEL_NAME}...')
    model = SentenceTransformer(MODEL_NAME)
    texts_to_encode = [t[:1500] for t in df_all['text'].tolist()]
    t0 = time.perf_counter()
    all_embeddings = model.encode(texts_to_encode, batch_size=256, show_progress_bar=True,
                                  normalize_embeddings=True)
    elapsed = time.perf_counter() - t0
    print(f'Encoded in {elapsed:.1f}s ({len(df_all)/elapsed:.0f} speeches/s)')
    np.savez_compressed(EMB_CACHE, embeddings=all_embeddings)

D = all_embeddings.shape[1]
# Extract filtered embeddings using original indices
embeddings = all_embeddings[df['orig_idx'].values]
print(f'D={D}, filtered embeddings: {embeddings.shape}')

# Daily aggregation per gender
df['day'] = df['date'].dt.normalize()
emb_cols = [f'e{i}' for i in range(D)]

df_emb = df[['day', 'speaker_gender']].copy()
for i in range(D):
    df_emb[f'e{i}'] = embeddings[:, i]

daily_list = []
for gender, eid in [('F', 1), ('M', 2)]:
    mask = df_emb['speaker_gender'] == gender
    g_daily = df_emb[mask].groupby('day')[emb_cols].mean().reset_index()
    g_daily['gender'] = gender
    g_daily['entity_id'] = eid
    g_daily['n_speeches'] = df_emb[mask].groupby('day').size().values
    daily_list.append(g_daily)

daily = pd.concat(daily_list, ignore_index=True).sort_values(['day', 'gender'])
daily['unix_ts'] = daily['day'].apply(lambda x: int(x.timestamp()))

print(f'\nDaily aggregated: {len(daily):,} (gender × day) points')
print(f'  Female days: {(daily["gender"]=="F").sum()}, Male days: {(daily["gender"]=="M").sum()}')

Loaded cached embeddings: (73631, 384)
D=384, filtered embeddings: (32291, 384)



Daily aggregated: 890 (gender × day) points
  Female days: 445, Male days: 445


In [4]:
# Build CVX index
INDEX_PATH = f'{CACHE_DIR}/parlamint_index.cvx'

if os.path.exists(INDEX_PATH):
    os.remove(INDEX_PATH)

index = cvx.TemporalIndex(m=16, ef_construction=200)
entity_ids = daily['entity_id'].values.astype(np.uint64)
timestamps = daily['unix_ts'].values.astype(np.int64)
vectors = daily[emb_cols].values.astype(np.float32)

t0 = time.perf_counter()
n = index.bulk_insert(entity_ids, timestamps, vectors, ef_construction=64)
elapsed = time.perf_counter() - t0
print(f'Inserted {n:,} points in {elapsed:.2f}s')
index.save(INDEX_PATH)

# Extract trajectories
traj_female = index.trajectory(entity_id=1)
traj_male = index.trajectory(entity_id=2)
print(f'Female trajectory: {len(traj_female)} days')
print(f'Male trajectory: {len(traj_male)} days')
print(f'Range: {pd.Timestamp(traj_female[0][0], unit="s").date()} to {pd.Timestamp(traj_female[-1][0], unit="s").date()}')

Inserted 890 points in 0.25s
Female trajectory: 445 days
Male trajectory: 445 days
Range: 2015-01-20 to 2023-02-23


---
## 3. Rhetorical Anchoring

Define **8 rhetorical anchors** in Spanish that capture dimensions of parliamentary discourse.
Project each gender's trajectory into this interpretable coordinate system.

In [5]:
# Rhetorical anchors in Spanish (matching the corpus language)
ANCHORS = {
    'ataque_personal': [
        'Usted miente y engaña a los ciudadanos con descaro',
        'Es usted un incompetente que no merece ocupar ese cargo',
        'Su señoría no tiene la menor idea de lo que está hablando',
    ],
    'politica_social': [
        'Debemos proteger a los más vulnerables con políticas sociales justas',
        'La sanidad pública y la educación son derechos fundamentales',
        'Es necesario aumentar las ayudas a las familias en situación de pobreza',
    ],
    'economia': [
        'El crecimiento económico y la creación de empleo son la prioridad',
        'La deuda pública y el déficit fiscal son insostenibles',
        'Necesitamos reformas estructurales para mejorar la competitividad',
    ],
    'emocional': [
        'Siento una profunda preocupación por el sufrimiento de las víctimas',
        'Es indignante lo que está ocurriendo con las personas más necesitadas',
        'Me emociona ver la solidaridad de los ciudadanos en estos momentos difíciles',
    ],
    'institucional': [
        'El respeto a la Constitución y al Estado de Derecho es innegociable',
        'Las instituciones democráticas deben ser protegidas y fortalecidas',
        'El diálogo parlamentario es la esencia de nuestra democracia',
    ],
    'territorial': [
        'La unidad de España no se puede poner en cuestión bajo ningún concepto',
        'Los derechos de las comunidades autónomas deben ser respetados',
        'El modelo territorial necesita una reforma profunda y consensuada',
    ],
    'genero': [
        'La igualdad entre hombres y mujeres es un derecho fundamental',
        'Debemos combatir la violencia machista con todos los instrumentos del Estado',
        'Las mujeres siguen sufriendo discriminación en el ámbito laboral y político',
    ],
    'seguridad': [
        'La seguridad ciudadana y la lucha contra el terrorismo son prioridad absoluta',
        'Las fuerzas y cuerpos de seguridad del Estado merecen nuestro apoyo total',
        'La inmigración irregular debe gestionarse con firmeza y humanidad',
    ],
}

ANCHOR_NAMES = list(ANCHORS.keys())
ANCHOR_LABELS = [n.replace('_', ' ').title() for n in ANCHOR_NAMES]

# Encode anchors
ANCHOR_CACHE = f'{CACHE_DIR}/parlamint_anchors.npz'

if os.path.exists(ANCHOR_CACHE):
    cached = np.load(ANCHOR_CACHE, allow_pickle=True)
    anchor_vectors = {name: cached[name] for name in ANCHOR_NAMES}
    print('Loaded cached anchor vectors')
else:
    from sentence_transformers import SentenceTransformer
    print(f'Encoding anchors with {MODEL_NAME}...')
    st_model = SentenceTransformer(MODEL_NAME)
    anchor_vectors = {}
    for name, phrases in ANCHORS.items():
        embs = st_model.encode(phrases, normalize_embeddings=True)
        anchor_vectors[name] = embs.mean(axis=0)
        print(f'  {name}: {len(phrases)} phrases')
    np.savez(ANCHOR_CACHE, **anchor_vectors)

anchor_list = [anchor_vectors[name].tolist() for name in ANCHOR_NAMES]
print(f'{len(ANCHOR_NAMES)} anchors ready')

# Project both trajectories
proj_female = cvx.project_to_anchors(traj_female, anchor_list, metric='cosine')
proj_male = cvx.project_to_anchors(traj_male, anchor_list, metric='cosine')
sum_female = cvx.anchor_summary(proj_female)
sum_male = cvx.anchor_summary(proj_male)

# Summary table
print(f'\nRhetorical proximity (1 - cosine distance) by gender:')
print(f'{"Anchor":20s} {"Female":>8s} {"Male":>8s} {"Gap":>8s} {"Direction":>12s}')
print('-' * 62)
for j, anchor in enumerate(ANCHOR_NAMES):
    f_prox = 1 - sum_female['mean'][j]
    m_prox = 1 - sum_male['mean'][j]
    gap = f_prox - m_prox
    direction = 'F closer' if gap > 0.002 else ('M closer' if gap < -0.002 else 'similar')
    print(f'{anchor:20s} {f_prox:8.4f} {m_prox:8.4f} {gap:+8.4f} {direction:>12s}')

Loaded cached anchor vectors
8 anchors ready

Rhetorical proximity (1 - cosine distance) by gender:
Anchor                 Female     Male      Gap    Direction
--------------------------------------------------------------
ataque_personal        0.4649   0.4892  -0.0244     M closer
politica_social        0.4375   0.4078  +0.0298     F closer
economia               0.4166   0.4004  +0.0161     F closer
emocional              0.4451   0.4311  +0.0140     F closer
institucional          0.6255   0.6406  -0.0150     M closer
territorial            0.5116   0.5150  -0.0034     M closer
genero                 0.3502   0.3281  +0.0220     F closer
seguridad              0.4118   0.4040  +0.0078     F closer


In [6]:
# Radar: rhetorical profile by gender
radar_closed = ANCHOR_LABELS + [ANCHOR_LABELS[0]]
f_prox = [1 - sum_female['mean'][j] for j in range(len(ANCHOR_NAMES))]
m_prox = [1 - sum_male['mean'][j] for j in range(len(ANCHOR_NAMES))]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=f_prox + [f_prox[0]], theta=radar_closed, fill='toself',
    name='Female MPs', line=dict(color=C_FEMALE, width=3),
    fillcolor='rgba(231, 76, 60, 0.15)',
))
fig.add_trace(go.Scatterpolar(
    r=m_prox + [m_prox[0]], theta=radar_closed, fill='toself',
    name='Male MPs', line=dict(color=C_MALE, width=3),
    fillcolor='rgba(52, 152, 219, 0.15)',
))
fig.update_layout(
    title='Rhetorical Profile — Female vs Male MPs (Spanish Parliament 2015-2023)',
    polar=dict(radialaxis=dict(visible=True), bgcolor='rgba(0,0,0,0)'),
    width=800, height=600, template=TEMPLATE,
)
fig.show()

---
## 4. Temporal Dynamics — Velocity, Changepoints, Small Multiples

In [7]:
# Small multiples: per-anchor proximity over time (30-day rolling average)
n_anchors = len(ANCHOR_NAMES)
f_dates = [pd.Timestamp(ts, unit='s') for ts, _ in traj_female]
m_dates = [pd.Timestamp(ts, unit='s') for ts, _ in traj_male]
f_dists = np.array([d for _, d in proj_female])
m_dists = np.array([d for _, d in proj_male])

# Truncate to common length
common_len = min(len(f_dates), len(m_dates))
f_dates_c, m_dates_c = f_dates[:common_len], m_dates[:common_len]
f_prox_t = 1.0 - f_dists[:common_len]
m_prox_t = 1.0 - m_dists[:common_len]

window = 30
f_smooth = pd.DataFrame(f_prox_t).rolling(window, center=True).mean().bfill().ffill().values
m_smooth = pd.DataFrame(m_prox_t).rolling(window, center=True).mean().bfill().ffill().values

rows = (n_anchors + 1) // 2
fig = make_subplots(rows=rows, cols=2, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=ANCHOR_LABELS, vertical_spacing=0.06)

for idx in range(n_anchors):
    row, col = idx // 2 + 1, idx % 2 + 1
    fig.add_trace(go.Scatter(x=f_dates_c, y=f_smooth[:, idx], mode='lines',
                             line=dict(color=C_FEMALE, width=2),
                             name='Female' if idx == 0 else None, showlegend=(idx == 0)),
                  row=row, col=col)
    fig.add_trace(go.Scatter(x=m_dates_c, y=m_smooth[:, idx], mode='lines',
                             line=dict(color=C_MALE, width=2),
                             name='Male' if idx == 0 else None, showlegend=(idx == 0)),
                  row=row, col=col)

# Event lines
date_range = [f_dates_c[0], f_dates_c[-1]]
for ev_date, ev_name in KEY_EVENTS.items():
    ev_ts = pd.Timestamp(ev_date)
    if date_range[0] <= ev_ts <= date_range[1]:
        for r in range(1, rows + 1):
            for c in [1, 2]:
                fig.add_vline(x=ev_ts, row=r, col=c,
                             line=dict(color=C_EVENT, width=0.5, dash='dot'))

fig.update_layout(
    title='Rhetorical Proximity Over Time — Female (red) vs Male (blue) MPs (30d avg)',
    height=200 * rows, width=1100, template=TEMPLATE,
)
fig.show()

In [8]:
# Velocity + changepoints for both genders
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Rhetorical Velocity — Female MPs', 'Rhetorical Velocity — Male MPs'],
                    vertical_spacing=0.08)

for gender, traj, proj, eid, color, row in [
    ('Female', traj_female, proj_female, 1, C_FEMALE, 1),
    ('Male', traj_male, proj_male, 2, C_MALE, 2),
]:
    # Velocity
    vels = []
    for i in range(1, len(proj) - 1):
        try:
            vel = cvx.velocity(proj, timestamp=proj[i][0])
            vels.append((traj[i][0], float(np.linalg.norm(vel))))
        except:
            continue
    
    if vels:
        dates = [pd.Timestamp(ts, unit='s') for ts, _ in vels]
        vals = [v for _, v in vels]
        s = pd.Series(vals, index=dates).rolling(14, center=True).mean()
        fig.add_trace(go.Scatter(x=dates, y=s.values, mode='lines',
                                 line=dict(color=color, width=1.5), name=gender),
                      row=row, col=1)
    
    # Changepoints — use raw timestamps
    proj_raw_ts = [(traj[i][0], proj[i][1]) for i in range(min(len(traj), len(proj)))]
    cps = cvx.detect_changepoints(eid, proj_raw_ts, penalty=0.5,
                                   min_segment_len=max(7, len(proj) // 30))
    for cp_ts, sev in cps:
        fig.add_trace(go.Scatter(
            x=[pd.Timestamp(cp_ts, unit='s')], y=[sev * max(vals) * 2],
            mode='markers', marker=dict(size=10, color=C_ACCENT, symbol='diamond'),
            showlegend=False), row=row, col=1)
    print(f'{gender}: {len(vels)} velocity points, {len(cps)} changepoints')

for ev_date, ev_name in KEY_EVENTS.items():
    ev_ts = pd.Timestamp(ev_date)
    for row in [1, 2]:
        fig.add_vline(x=ev_ts, row=row, col=1, line=dict(color=C_EVENT, width=1, dash='dot'))
    fig.add_annotation(x=ev_ts, y=1.02, yref='paper', text=ev_name,
                       showarrow=False, font=dict(size=7, color=C_EVENT), textangle=-35)

fig.update_layout(title='Rhetorical Velocity — When Does Parliamentary Discourse Change Fastest?',
                  height=600, width=1100, template=TEMPLATE)
fig.show()

Female: 443 velocity points, 1 changepoints
Male: 443 velocity points, 0 changepoints


---
## 5. Counterfactual — COVID Impact on Gender Gap

In [9]:
# Counterfactual: how did COVID change female MPs' rhetorical trajectory?
covid_ts = int(pd.Timestamp('2020-03-14').timestamp())
proj_raw_ts_f = [(traj_female[i][0], proj_female[i][1])
                  for i in range(min(len(traj_female), len(proj_female)))]

pre = [(ts, d) for ts, d in proj_raw_ts_f if ts < covid_ts]
post = [(ts, d) for ts, d in proj_raw_ts_f if ts >= covid_ts]
print(f'Counterfactual split at COVID (2020-03-14): pre={len(pre)}, post={len(post)}')

if len(pre) >= 20 and len(post) >= 10:
    result = cvx.counterfactual_trajectory(pre, post, covid_ts)
    print(f'Total divergence: {result["total_divergence"]:.4f}')
    print(f'Max divergence: {result["max_divergence_value"]:.4f} at '
          f'{pd.Timestamp(result["max_divergence_time"], unit="s").date()}')

    div_dates = [pd.Timestamp(ts, unit='s') for ts, _ in result['divergence_curve']]
    div_vals = [v for _, v in result['divergence_curve']]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=div_dates, y=div_vals, mode='lines+markers',
                             line=dict(color=C_FEMALE, width=2.5), marker=dict(size=3)))
    fig.add_vline(x=pd.Timestamp('2020-03-14'), line=dict(color=C_EVENT, width=2, dash='dash'))
    fig.add_annotation(x=pd.Timestamp('2020-03-14'), y=max(div_vals) * 0.9,
                       text='COVID State of Alarm', showarrow=True, arrowhead=2,
                       font=dict(color=C_EVENT))
    fig.update_layout(
        title='Counterfactual: How Much Did COVID Deviate Female MPs\' Rhetorical Trajectory?',
        xaxis_title='Date', yaxis_title='Divergence from Expected',
        height=400, width=1000, template=TEMPLATE)
    fig.show()

# Hurst exponent
h_f = cvx.hurst_exponent(proj_female)
h_m = cvx.hurst_exponent(proj_male)
print(f'\nHurst exponent — Female: {h_f:.3f}, Male: {h_m:.3f}')
print(f'  {"Both persistent" if h_f > 0.5 and h_m > 0.5 else "Mixed"}')

Counterfactual split at COVID (2020-03-14): pre=252, post=193
Total divergence: 15261137.1283
Max divergence: 0.3569 at 2021-06-24



Hurst exponent — Female: 0.679, Male: 0.677
  Both persistent


---
## 6. Party × Gender — Is Polarization Driven by Party or Gender?

In [10]:
# Build per-party×gender trajectories for top 4 parties
TOP_PARTIES = ['Partido Socialista Obrero Español', 'Partido Popular',
               'Ciudadanos', 'Vox']
PARTY_SHORT = {'Partido Socialista Obrero Español': 'PSOE', 'Partido Popular': 'PP',
               'Ciudadanos': 'Cs', 'Vox': 'Vox'}

party_trajs = {}
party_sigs = {}

for party in TOP_PARTIES:
    short = PARTY_SHORT[party]
    for gender in ['F', 'M']:
        key = f'{short}_{gender}'
        mask = (df['speaker_party'] == party) & (df['speaker_gender'] == gender)
        sub = df[mask].copy()
        if len(sub) < 50:
            print(f'{key}: skipped ({len(sub)} speeches)')
            continue
        
        # Use contiguous df index to get embeddings
        sub_embs = embeddings[sub.index.values]
        
        # Monthly aggregation
        sub['month'] = sub['date'].dt.to_period('M').dt.to_timestamp()
        months_list = sorted(sub['month'].unique())
        traj = []
        for m in months_list:
            m_mask = (sub['month'] == m).values
            m_embs = sub_embs[m_mask]
            if len(m_embs) > 0:
                mean_emb = m_embs.mean(axis=0).tolist()
                ts = int(m.timestamp())
                traj.append((ts, mean_emb))
        
        if len(traj) >= 20:
            party_trajs[key] = traj
            proj = cvx.project_to_anchors(traj, anchor_list, metric='cosine')
            sig = cvx.path_signature(proj, depth=2, time_augmentation=True)
            party_sigs[key] = sig
            print(f'{key}: {len(traj)} months, {len(sub)} speeches')

# Signature distance matrix
keys = sorted(party_sigs.keys())
n_k = len(keys)

if n_k < 2:
    print(f'Only {n_k} party×gender combos with enough data — skipping distance analysis')
else:
    sig_matrix = np.zeros((n_k, n_k))
    for i in range(n_k):
        for j in range(n_k):
            sig_matrix[i, j] = cvx.signature_distance(party_sigs[keys[i]], party_sigs[keys[j]])
    
    within_gender, within_party, between = [], [], []
    for i in range(n_k):
        for j in range(i + 1, n_k):
            pi, gi = keys[i].rsplit('_', 1)
            pj, gj = keys[j].rsplit('_', 1)
            d = sig_matrix[i, j]
            if pi == pj:
                within_party.append(d)
            elif gi == gj:
                within_gender.append(d)
            else:
                between.append(d)
    
    print(f'\nSignature distance analysis:')
    if within_party:
        print(f'  Within same party (F vs M): {np.mean(within_party):.3f} ± {np.std(within_party):.3f}')
    if within_gender:
        print(f'  Within same gender (cross-party): {np.mean(within_gender):.3f} ± {np.std(within_gender):.3f}')
    if between:
        print(f'  Between (diff party + diff gender): {np.mean(between):.3f} ± {np.std(between):.3f}')
    
    if within_party and within_gender:
        driver = 'PARTY' if np.mean(within_party) < np.mean(within_gender) else 'GENDER'
        print(f'  → Rhetorical similarity is driven more by {driver}')
    
    fig = go.Figure(go.Heatmap(
        z=sig_matrix, x=keys, y=keys,
        colorscale='Viridis', text=np.round(sig_matrix, 2), texttemplate='%{text}',
    ))
    fig.update_layout(
        title='Rhetorical Trajectory Similarity (Party × Gender)',
        height=500, width=600, template=TEMPLATE)
    fig.show()

PSOE_F: 75 months, 3155 speeches
PSOE_M: 78 months, 3017 speeches
PP_F: 76 months, 2422 speeches
PP_M: 79 months, 3902 speeches
Cs_F: 63 months, 857 speeches
Cs_M: 68 months, 1548 speeches
Vox_F: 37 months, 605 speeches
Vox_M: 38 months, 1267 speeches

Signature distance analysis:
  Within same party (F vs M): 1.084 ± 0.203
  Within same gender (cross-party): 1.231 ± 0.470
  Between (diff party + diff gender): 1.418 ± 0.582
  → Rhetorical similarity is driven more by PARTY


---
## 7. Classification — Predict Speaker Gender from Rhetorical Features

In [11]:
# Feature extraction per (gender × quarter)
# Uses raw trajectory timestamps for temporal filtering
quarters = pd.date_range('2015-01-01', '2023-06-01', freq='QS')
feature_rows = []

for gender, traj, proj, eid in [('F', traj_female, proj_female, 1),
                                  ('M', traj_male, proj_male, 2)]:
    proj_raw_ts = [(traj[i][0], proj[i][1]) for i in range(min(len(traj), len(proj)))]
    
    for q_start in quarters[:-1]:
        q_end = q_start + pd.DateOffset(months=3)
        q_start_ts, q_end_ts = int(q_start.timestamp()), int(q_end.timestamp())
        
        q_proj = [(ts, d) for ts, d in proj_raw_ts if q_start_ts <= ts < q_end_ts]
        q_traj = [(ts, v) for ts, v in traj if q_start_ts <= ts < q_end_ts]
        
        if len(q_proj) < 7:
            continue
        
        summary = cvx.anchor_summary(q_proj)
        tf = cvx.temporal_features(q_traj)
        
        try:
            h = cvx.hurst_exponent(q_proj)
        except:
            h = 0.5
        
        ts_list = [ts for ts, _ in q_traj]
        try:
            ef = cvx.event_features(ts_list)
        except:
            ef = {'burstiness': 0, 'memory': 0, 'temporal_entropy': 0}
        
        feat = {
            'gender': gender, 'label': 1 if gender == 'F' else 0,
            'quarter': q_start, 'n_points': len(q_proj),
            'hurst': h, 'burstiness': ef.get('burstiness', 0),
            'memory': ef.get('memory', 0),
        }
        for j, anchor in enumerate(ANCHOR_NAMES):
            feat[f'anchor_mean_{anchor}'] = summary['mean'][j]
            feat[f'anchor_trend_{anchor}'] = summary['trend'][j]
        for i, v in enumerate(tf[:10]):
            feat[f'tf_{i}'] = v
        feature_rows.append(feat)

df_feat = pd.DataFrame(feature_rows)
print(f'Feature matrix: {df_feat.shape}')
print(f'Female: {(df_feat["label"]==1).sum()}, Male: {(df_feat["label"]==0).sum()}')

# Temporal split: train 2015-2020, test 2021-2023
split_date = pd.Timestamp('2021-01-01')
train_mask = df_feat['quarter'] < split_date
test_mask = df_feat['quarter'] >= split_date

meta_cols = ['gender', 'label', 'quarter', 'n_points']
feat_cols = [c for c in df_feat.columns if c not in meta_cols]

X_train = np.nan_to_num(df_feat[train_mask][feat_cols].values.astype(float))
y_train = df_feat[train_mask]['label'].values
X_test = np.nan_to_num(df_feat[test_mask][feat_cols].values.astype(float))
y_test = df_feat[test_mask]['label'].values

print(f'\nTrain: {len(X_train)} (2015-2020), Test: {len(X_test)} (2021-2023)')

if len(X_train) > 3 and len(X_test) > 3:
    scaler = StandardScaler()
    clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    clf.fit(scaler.fit_transform(X_train), y_train)
    
    y_pred = clf.predict(scaler.transform(X_test))
    y_prob = clf.predict_proba(scaler.transform(X_test))[:, 1]
    
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    print(f'\n=== CVX Rhetorical Features → Speaker Gender ===')
    print(f'  F1: {f1:.3f}, AUC: {auc:.3f}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Male", "Female"])}')
    
    importance = pd.DataFrame({
        'feature': feat_cols, 'coef': clf.coef_[0],
    }).sort_values('coef', key=abs, ascending=False)
    
    fig = go.Figure(go.Bar(
        x=importance.head(15)['coef'].values,
        y=importance.head(15)['feature'].values,
        orientation='h',
        marker_color=[C_FEMALE if c > 0 else C_MALE for c in importance.head(15)['coef']],
    ))
    fig.update_layout(
        title='Top 15 Features for Predicting Speaker Gender (positive = predicts Female)',
        height=500, width=900, template=TEMPLATE, yaxis=dict(autorange='reversed'))
    fig.show()

Feature matrix: (54, 33)
Female: 27, Male: 27

Train: 38 (2015-2020), Test: 16 (2021-2023)

=== CVX Rhetorical Features → Speaker Gender ===
  F1: 0.941, AUC: 1.000

              precision    recall  f1-score   support

        Male       1.00      0.88      0.93         8
      Female       0.89      1.00      0.94         8

    accuracy                           0.94        16
   macro avg       0.94      0.94      0.94        16
weighted avg       0.94      0.94      0.94        16



---
## Summary

### Dataset

**ParlaMint-ES v5.0** — 32,541 speeches from the Spanish Parliament (2015-2023),
355 female MPs + 486 male MPs, CC-BY 4.0.

### CVX functions used

| Function | Section | Signal |
|----------|---------|--------|
| `project_to_anchors`, `anchor_summary` | §3, §4 | Rhetorical proximity per dimension |
| `velocity`, `detect_changepoints` | §4 | When rhetoric shifts |
| `counterfactual_trajectory` | §5 | COVID impact quantification |
| `hurst_exponent` | §5 | Persistent vs reactive rhetoric |
| `path_signature`, `signature_distance` | §6 | Party × gender trajectory fingerprints |
| `temporal_features`, `event_features` | §7 | Feature extraction for classification |

### Key insight

ParlaMint provides **real temporal data** (8 years, daily resolution) for studying how
political discourse differs by gender — using CVX as the analytical engine. The anchors
are defined in Spanish to maximize semantic alignment with the corpus.

Unlike B7 (synthetic data with perfect separation), real parliamentary data will show
subtler, noisier patterns — which makes the temporal analytics more valuable because
they capture **dynamics** that static content analysis misses.